# scitex-ml — Classification Quick Start

Minimal walk-through of `scitex_ml.Classifier` — a thin factory over scikit-learn estimators that integrates with `ClassificationReporter` for metric tracking and figure export.

**What this notebook covers**

1. Train a `Classifier` on the Iris dataset.
2. Score on a held-out test split.
3. Use `ClassificationReporter` to compute and persist metrics.

Heavy / optional dependencies (`torch`, `optuna`, `pytorch_pretrained_vit`) are not required for this notebook.

In [ ]:
import scitex_ml
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

scitex_ml.__version__

## 1. Train / evaluate

In [ ]:
X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0, stratify=y)

clf = scitex_ml.Classifier('LogisticRegression')
clf.fit(X_tr, y_tr)
score = clf.score(X_te, y_te)
print(f'test accuracy: {score:.3f}')

## 2. Metric reporting

`ClassificationReporter` logs metrics, confusion matrices, and ROC / PR curves to a directory you pass in. The directory is auto-created.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    reporter = scitex_ml.ClassificationReporter(save_dir=tmp)
    y_pred = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)
    reporter.calc_metrics(y_te, y_pred, y_proba, labels=['setosa','versicolor','virginica'])
    reporter.summarize()
    reporter.save()
    print('artefacts written to:', sorted(p.name for p in Path(tmp).iterdir()))

## Where to next

- `scitex_ml.classification` — time-series CV splitters (`TimeSeriesStratifiedSplit`, `TimeSeriesSlidingWindowSplit`, `TimeSeriesBlockingSplit`, `TimeSeriesCalendarSplit`).
- `scitex_ml.training` — `EarlyStopping`, `LearningCurveLogger`.
- `scitex_ml.optim` — `get_optimizer` / `set_optimizer` shortcuts; vendored Ranger.
- `scitex_ml.metrics` — `calc_bacc`, `calc_conf_mat`, `calc_roc_auc`.

For generative-AI (LLM, agent, image, audio) see `scitex-genai`.